# Render the full kiisafezone README showcase 🎬

Render the complete portrait composition from `assets/macos-readme-showcase.svg` using the Rust-based `svg-to-video` CLI. The source SVG is fetched from the latest `main` branch in this notebook.

**Target output:** 1080 × 1920 · 30 FPS · 59 seconds · H.264 MP4.

Run the cells from top to bottom. The renderer installation is needed only once per fresh Colab runtime.

In [ ]:
# Install the native build tools and FFmpeg (do not install Ubuntu's Snap Chromium stub)
!apt-get update -qq
!apt-get install -y -qq build-essential gcc g++ make pkg-config libssl-dev ffmpeg curl ca-certificates

# Install the official Google Chrome .deb so headless_chrome can find a real browser
!curl -fsSL https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -o /tmp/google-chrome-stable.deb
!apt-get install -y -qq /tmp/google-chrome-stable.deb

# Install Rust and explicitly preserve standard system paths
!curl https://sh.rustup.rs -sSf | sh -s -- -y
%env PATH=/root/.cargo/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin

# Confirm compiler, browser, and FFmpeg before compiling the renderer
!command -v cc && cc --version | head -n 1
!command -v google-chrome-stable && google-chrome-stable --version
!command -v ffmpeg && ffmpeg -version | head -n 1

# Compile svg-to-video; limit parallel jobs to keep Colab resource usage predictable
!cargo install svg-to-video --locked --jobs 2
!command -v svg-to-video && svg-to-video --help

In [ ]:
# Fetch the latest showcase source from the repository's main branch
%cd /content
!rm -rf /content/kiisafezone
!git clone --depth 1 --branch main https://github.com/kiisafezone/kiisafezone.git
%cd /content/kiisafezone
!git log -1 --oneline
!ls -lh assets/macos-readme-showcase.svg

In [ ]:
# Validate the actual source SVG before rendering
from pathlib import Path
import xml.etree.ElementTree as ET

svg_path = Path('/content/kiisafezone/assets/macos-readme-showcase.svg')
assert svg_path.is_file(), f'Missing source SVG: {svg_path}'
root = ET.parse(svg_path).getroot()
print('Source:', svg_path)
print('Canvas:', root.attrib.get('width'), '×', root.attrib.get('height'))
print('ViewBox:', root.attrib.get('viewBox'))
print('File size:', f'{svg_path.stat().st_size:,}', 'bytes')
assert root.attrib.get('width') == '1080', 'Unexpected SVG width'
assert root.attrib.get('height') == '1920', 'Unexpected SVG height'
assert root.attrib.get('viewBox') == '0 0 1080 1920', 'Unexpected SVG viewBox'
print('Preflight passed.')

In [ ]:
# Render the complete 59-second README showcase
%cd /content/kiisafezone
!mkdir -p out
!rm -f out/showcase.mp4
!svg-to-video video assets/macos-readme-showcase.svg out/showcase.mp4 --duration 59 --fps 30 --width 1080 --height 1920 --codec h264
!ls -lh out/showcase.mp4

In [ ]:
# Verify the rendered video's dimensions, frame rate, and duration
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate -show_entries format=duration -of default=noprint_wrappers=1 out/showcase.mp4

In [ ]:
# Download the finished showcase MP4
from google.colab import files
files.download('/content/kiisafezone/out/showcase.mp4')